# RAGFlow PDF 前端渲染完整指南

本指南详细讲解从用户点击查看 PDF 参考文献，到前端完整渲染 PDF 并高亮相关 chunks 的全过程。

## 🎯 核心流程

```
用户点击参考文献
    ↓
触发 PdfSheet 组件显示
    ↓
构建 PDF URL: /api/v1/documents/{doc_id}/preview
    ↓
后端返回二进制 PDF 数据
    ↓
PdfPreview 组件使用 react-pdf-highlighter 渲染
    ↓
高亮相关 chunks 的位置
```

## 第 1 步：Hook 层 - 获取文档 URL 和高亮数据

**文件位置**: `web/src/hooks/use-document-request.ts`

### 1.1 获取文档 URL

```typescript
// Hook: useGetDocumentUrl
export const useGetDocumentUrl = (documentId?: string) => {
  const getDocumentUrl = useCallback(
    (id?: string) => {
      // 构建 URL: {API_BASE}/documents/{document_id}/preview
      return `${restAPIv1}/documents/${id || documentId}/preview`;
    },
    [documentId],
  );

  return getDocumentUrl;
};
```

**功能**: 
- 生成 PDF 预览 URL
- 返回一个函数，可以根据 document ID 动态生成 URL

### 1.2 获取高亮数据

```typescript
// Hook: useGetChunkHighlights
export const useGetChunkHighlights = (
  selectedChunk: IChunk | IReferenceChunk,
) => {
  const [size, setSize] = useState({ width: 849, height: 1200 });

  // 根据 chunk 数据构建高亮坐标
  const highlights: IHighlight[] = useMemo(() => {
    return buildChunkHighlights(selectedChunk, size);
  }, [selectedChunk, size]);

  // 动态更新 PDF 页面尺寸
  const setWidthAndHeight = (width: number, height: number) => {
    setSize((pre) => {
      if (pre.height !== height || pre.width !== width) {
        return { height, width };
      }
      return pre;
    });
  };

  return { highlights, setWidthAndHeight };
};
```

**功能**:
- `buildChunkHighlights`: 将 chunk 的坐标信息转换为 PDF 高亮位置
- `setWidthAndHeight`: 当 PDF 加载时更新页面尺寸（用于精确定位高亮）

## 第 2 步：组件层 - PdfSheet 容器

**文件位置**: `web/src/components/pdf-drawer/index.tsx`

```typescript
interface IProps extends IModalProps<any> {
  documentId: string;              // 文档 ID
  chunk: IChunk | IReferenceChunk; // 要高亮的 chunk 数据
  width?: string | number;         // 模态框宽度
  height?: string | number;        // 模态框高度
}

export const PdfSheet = ({
  hideModal,           // 关闭模态框的回调
  documentId,          // 文档 ID
  chunk,               // 选中的 chunk
  width = '50vw',
  height,
}: IProps) => {
  // 第 1 步：获取生成 URL 的函数
  const getDocumentUrl = useGetDocumentUrl(documentId);
  
  // 第 2 步：生成 PDF URL
  const url = getDocumentUrl(documentId);
  // 结果: "http://localhost:8000/api/v1/documents/{doc_id}/preview"
  
  // 第 3 步：获取高亮数据
  const { highlights, setWidthAndHeight } = useGetChunkHighlights(chunk);
  // highlights 是一个数组，包含 chunk 在 PDF 中的坐标信息
  // [
  //   {
  //     position: { pageNumber: 1, x: 100, y: 200, width: 150, height: 50 },
  //     comment: { emoji: '📌', text: 'Important chunk' }
  //   },
  //   ...
  // ]

  return (
    <Sheet open onOpenChange={hideModal}>
      <SheetContent
        className={cn(`max-w-full`)}
        style={{ width: width, height: height ? height : undefined }}
      >
        <SheetHeader>
          <SheetTitle>Document Previewer</SheetTitle>
        </SheetHeader>
        
        {/* 只有当 URL 和 documentId 都存在时才渲染 */}
        {url && documentId && (
          <PdfPreview
            className={'p-0 !h-[calc(100vh-80px)] w-full'}
            highlights={highlights}           // 传递高亮数据
            setWidthAndHeight={setWidthAndHeight}  // 传递尺寸更新函数
            url={url}                         // 传递 PDF URL
          />
        )}
      </SheetContent>
    </Sheet>
  );
};
```

**关键点**:
1. ✅ 调用 `useGetDocumentUrl` 生成 URL
2. ✅ 调用 `useGetChunkHighlights` 获取高亮数据
3. ✅ 将这些数据传递给 `PdfPreview` 组件

## 第 3 步：渲染层 - PdfPreview 组件

**文件位置**: `web/src/components/document-preview/pdf-preview.tsx`

这是最核心的渲染组件！

```typescript
interface IProps {
  highlights?: IHighlight[];  // 要高亮的位置
  setWidthAndHeight?: (width: number, height: number) => void;  // 尺寸回调
  url: string;                // PDF URL
  className?: string;         // CSS 类名
}

const PdfPreview = ({
  highlights: state,          // 重命名为 state
  setWidthAndHeight,
  url,
  className,
}: IProps) => {
  // 用于滚动到第一个高亮的引用
  const ref = useRef<(highlight: IHighlight) => void>(() => {});
  
  // 处理加载错误
  const error = useCatchDocumentError(url);

  // 当高亮数据改变时，自动滚动到第一个高亮
  useEffect(() => {
    let timer = null;
    if (state?.length && state?.length > 0) {
      timer = setTimeout(() => {
        ref?.current(state[0]);  // 滚动到第一个高亮
      }, 100);
    }
    return () => {
      if (timer) clearTimeout(timer);
    };
  }, [state]);

  // 设置认证 header
  const httpHeaders = {
    [Authorization]: getAuthorization(),  // 从 token 获取认证信息
  };

  return (
    <div className={cn('relative size-full rounded overflow-hidden', className)}>
      {/* PdfLoader: 从 URL 加载 PDF 文件 */}
      <Loader
        url={url}                    // PDF URL
        httpHeaders={httpHeaders}    // 认证 header
        beforeLoad={
          <div className="absolute inset-0 flex items-center justify-center">
            <Spin />  {/* 加载中显示旋转图标 */}
          </div>
        }
        workerSrc="/pdfjs-dist/pdf.worker.min.js"  // PDF.js worker
        errorMessage={<FileError>{error}</FileError>}  // 错误提示
      >
        {(pdfDocument) => {
          // 当 PDF 加载完成后调用此函数
          // pdfDocument 是 PDF.js 的 PDFDocument 对象
          
          // 获取第一页以计算尺寸
          pdfDocument.getPage(1).then((page) => {
            const viewport = page.getViewport({ scale: 1 });
            const width = viewport.width;   // PDF 页面宽度（pt）
            const height = viewport.height; // PDF 页面高度（pt）
            
            // 告诉父组件 PDF 的尺寸
            setWidthAndHeight?.(width, height);
          });

          return (
            <PdfHighlighter
              pdfDocument={pdfDocument}
              enableAreaSelection={(event) => event.altKey}  // Alt + 拖拽可以框选
              onScrollChange={resetHash}
              scrollRef={(scrollTo) => {
                ref.current = scrollTo;  // 保存滚动函数
              }}
              onSelectionFinished={() => null}
              
              // 自定义高亮的渲染
              highlightTransform={(
                highlight,
                index,
                setTip,
                hideTip,
                viewportToScaled,
                screenshot,
                isScrolledTo,
              ) => {
                // 判断是文本高亮还是区域高亮
                const isTextHighlight = !(
                  highlight.content && highlight.content.image
                );

                // 渲染高亮
                const component = isTextHighlight ? (
                  <Highlight
                    isScrolledTo={isScrolledTo}
                    position={highlight.position}  // 高亮位置
                    comment={highlight.comment}    // 高亮注释
                  />
                ) : (
                  <AreaHighlight
                    isScrolledTo={isScrolledTo}
                    highlight={highlight}
                    onChange={() => {}}
                  />
                );

                return (
                  <Popup
                    popupContent={<HighlightPopup {...highlight} />}
                    onMouseOver={(popupContent) =>
                      setTip(highlight, () => popupContent)  // 显示 tooltip
                    }
                    onMouseOut={hideTip}
                    key={index}
                  >
                    {component}
                  </Popup>
                );
              }}
              highlights={state || []}  // 传递高亮数据
            />
          );
        }}
      </Loader>
    </div>
  );
};

export default memo(PdfPreview);
```

**核心步骤**:
1. ✅ `PdfLoader` 从 URL 加载 PDF（返回二进制数据）
2. ✅ 计算 PDF 页面尺寸
3. ✅ `PdfHighlighter` 使用 PDF.js 渲染 PDF
4. ✅ 在指定位置绘制高亮

## 第 4 步：后端 API 返回二进制数据

**文件位置**: `api/apps/restful_apis/document_api.py`

```python
@manager.route("/documents/<doc_id>/preview", methods=["GET"])
@login_required
async def get(doc_id):
    """
    返回 PDF 文件的原始二进制数据
    
    请求格式:
    GET /api/v1/documents/{doc_id}/preview
    Headers: Authorization: Bearer {token}
    
    响应:
    - 状态码: 200
    - Content-Type: application/pdf
    - Body: 二进制 PDF 数据
    """
    try:
        # 1️⃣ 从数据库查询文档元数据
        e, doc = DocumentService.get_by_id(doc_id)
        if not e:
            return get_data_error_result(message="Document not found!")
        
        # 2️⃣ 获取存储位置 (bucket, location)
        b, n = File2DocumentService.get_storage_address(doc_id=doc_id)
        
        # 3️⃣ 从对象存储（MinIO/S3）获取二进制数据
        data = await thread_pool_exec(settings.STORAGE_IMPL.get, b, n)
        
        # 4️⃣ 创建 HTTP 响应
        response = await make_response(data)
        
        # 5️⃣ 设置正确的 Content-Type
        ext = re.search(r"\.([^.]+)$", doc.name.lower())
        ext = ext.group(1) if ext else None
        content_type = None
        if ext:
            fallback_prefix = "image" if doc.type == FileType.VISUAL.value else "application"
            content_type = CONTENT_TYPE_MAP.get(ext, f"{fallback_prefix}/{ext}")
        
        apply_safe_file_response_headers(response, content_type, ext)
        
        # 6️⃣ 返回二进制数据！
        return response
    
    except Exception as e:
        return server_error_response(e)
```

**关键点**:
- 📝 从数据库查询元数据（文件名、类型等）
- 📦 从对象存储获取二进制数据
- 🎯 设置正确的 `Content-Type` header
- ✅ 返回原始二进制，不做任何转换

## 第 5 步：数据流全景图

### 5.1 数据结构

#### Chunk 数据格式
```typescript
interface IReferenceChunk {
  chunk_id: string;           // chunk ID
  doc_id: string;             // 文档 ID
  content: string;            // chunk 内容
  positions?: Array<{         // ⭐️ chunk 在 PDF 中的位置
    page_number: number;      // 页码（1-indexed）
    bbox: {
      x: number;              // 坐标 X（相对于页面）
      y: number;              // 坐标 Y（相对于页面）
      w: number;              // 宽度
      h: number;              // 高度
    };
  }>;
}
```

#### 高亮数据格式（由 buildChunkHighlights 生成）
```typescript
interface IHighlight {
  position: {
    pageNumber: number;       // 页码
    x: number;                // 0-1 之间的相对位置
    y: number;                // 0-1 之间的相对位置
    width: number;            // 宽度
    height: number;           // 高度
  };
  comment: {
    emoji: string;            // 高亮图标
    text: string;             // 高亮文本
  };
  content?: {
    text: string;             // 高亮的文本内容
    image?: string;           // 如果是图片高亮
  };
}
```

### 5.2 请求/响应流

```
┌─────────────────────────────────────────────────────────────────┐
│ 前端浏览器                                                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                   │
│ 用户点击参考文献按钮                                               │
│         ↓                                                         │
│ 触发 PdfSheet 组件                                               │
│         ↓                                                         │
│ 调用 useGetDocumentUrl(doc_id)                                  │
│   → 返回: "/api/v1/documents/{doc_id}/preview"                 │
│         ↓                                                         │
│ 调用 useGetChunkHighlights(chunk)                               │
│   → 返回: [                                                      │
│       {                                                           │
│         position: { pageNumber: 1, x: 0.1, y: 0.2, ... },     │
│         comment: { emoji: '📌', text: 'chunk content' }        │
│       },                                                          │
│       ...                                                         │
│     ]                                                             │
│         ↓                                                         │
│ 传递到 PdfPreview 组件                                           │
│         ↓                                                         │
│ PdfLoader 开始加载 PDF                                          │
│         ↓                                                         │
│ 📡 HTTP GET 请求: /api/v1/documents/{doc_id}/preview           │
│    Headers: Authorization: Bearer {token}                        │
│                                                                   │
└─────────────────────────────────────────────────────────────────┘
                          ↓↓↓ 网络请求 ↓↓↓
┌─────────────────────────────────────────────────────────────────┐
│ 后端服务器                                                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                   │
│ 1️⃣ 收到请求: GET /api/v1/documents/{doc_id}/preview            │
│         ↓                                                         │
│ 2️⃣ 验证认证 (login_required)                                   │
│         ↓                                                         │
│ 3️⃣ DocumentService.get_by_id(doc_id)                          │
│    从数据库查询文档元数据                                        │
│         ↓                                                         │
│ 4️⃣ File2DocumentService.get_storage_address(doc_id)           │
│    获取对象存储位置: (bucket, location)                        │
│         ↓                                                         │
│ 5️⃣ settings.STORAGE_IMPL.get(bucket, location)                │
│    从 MinIO/S3 获取二进制数据                                   │
│         ↓                                                         │
│ 6️⃣ 设置 Content-Type: application/pdf                        │
│         ↓                                                         │
│ 7️⃣ 返回响应                                                     │
│    Content-Type: application/pdf                               │
│    Body: [PDF 二进制数据]                                       │
│                                                                   │
└─────────────────────────────────────────────────────────────────┘
                          ↓↓↓ 响应返回 ↓↓↓
┌─────────────────────────────────────────────────────────────────┐
│ 前端浏览器（继续）                                               │
├─────────────────────────────────────────────────────────────────┤
│                                                                   │
│ PdfLoader 接收二进制响应                                        │
│         ↓                                                         │
│ 调用回调函数: (pdfDocument) => {                               │
│   // pdfDocument 是 PDF.js 的 PDFDocument 对象                 │
│   // 已经自动解析了二进制数据                                  │
│         ↓                                                         │
│   PdfHighlighter 开始渲染                                      │
│         ↓                                                         │
│   绘制 PDF 页面                                                │
│         ↓                                                         │
│   在高亮位置绘制 Highlight 组件                                 │
│         ↓                                                         │
│   用户可以看到 PDF 和高亮的 chunks                             │
│ }                                                                 │
│         ↓                                                         │
│ 用户点击高亮 → 显示 Tooltip                                    │
│         ↓                                                         │
│ 用户关闭 → 调用 hideModal()                                   │
│                                                                   │
└─────────────────────────────────────────────────────────────────┘
```

## 第 6 步：完整实现示例（你的 RAG 系统）

### 前端部分

#### 6.1 创建你的 Hook
```typescript
// hooks/useDocumentPreview.ts
import { useCallback, useState, useMemo } from 'react';

export const useDocumentPreview = (documentId?: string) => {
  // 生成 PDF URL
  const getDocumentUrl = useCallback(
    (id?: string) => {
      // 你的 API 端点
      return `${import.meta.env.VITE_API_URL}/documents/${id || documentId}/preview`;
    },
    [documentId],
  );

  return { getDocumentUrl };
};

export const useChunkHighlights = (chunk: any) => {
  const [size, setSize] = useState({ width: 849, height: 1200 });

  // 根据 chunk 坐标构建高亮
  const highlights = useMemo(() => {
    if (!chunk?.positions) return [];
    
    return chunk.positions.map((pos: any) => ({
      position: {
        pageNumber: pos.page_number,
        x: pos.bbox.x / size.width,      // 转换为 0-1 相对位置
        y: pos.bbox.y / size.height,
        width: pos.bbox.w / size.width,
        height: pos.bbox.h / size.height,
      },
      comment: {
        emoji: '📌',
        text: chunk.content.substring(0, 100), // 前 100 字符
      },
    }));
  }, [chunk, size]);

  const setWidthAndHeight = (width: number, height: number) => {
    setSize({ width, height });
  };

  return { highlights, setWidthAndHeight };
};
```

#### 6.2 创建 PDF 预览组件
```typescript
// components/PDFViewer.tsx
import React, { useCallback, useEffect, useRef } from 'react';
import { PdfLoader, PdfHighlighter, Highlight, AreaHighlight, Popup } from 'react-pdf-highlighter';
import 'react-pdf-highlighter/style/pdf-highlighter.css';

interface PDFViewerProps {
  url: string;
  highlights: any[];
  onHeightChange?: (w: number, h: number) => void;
}

export const PDFViewer: React.FC<PDFViewerProps> = ({
  url,
  highlights,
  onHeightChange,
}) => {
  const scrollRef = useRef<any>(null);
  const [authToken] = useAuthToken();  // 获取认证 token

  const httpHeaders = {
    Authorization: `Bearer ${authToken}`,
  };

  useEffect(() => {
    if (highlights.length > 0 && scrollRef.current) {
      // 自动滚动到第一个高亮
      setTimeout(() => scrollRef.current(highlights[0]), 100);
    }
  }, [highlights]);

  return (
    <div style={{ width: '100%', height: '100%' }}>
      <PdfLoader
        url={url}
        httpHeaders={httpHeaders}
        beforeLoad={<div>Loading PDF...</div>}
        workerSrc="/pdfjs-dist/pdf.worker.min.js"
      >
        {(pdfDocument) => {
          // 获取页面尺寸
          pdfDocument.getPage(1).then((page) => {
            const viewport = page.getViewport({ scale: 1 });
            onHeightChange?.(viewport.width, viewport.height);
          });

          return (
            <PdfHighlighter
              pdfDocument={pdfDocument}
              scrollRef={(scrollTo) => (scrollRef.current = scrollTo)}
              highlights={highlights}
              highlightTransform={(highlight, index, setTip, hideTip) => {
                const isTextHighlight = !(
                  highlight.content && highlight.content.image
                );

                return (
                  <Popup
                    popupContent={<div>{highlight.comment?.text}</div>}
                    onMouseOver={() => setTip(highlight, () => null)}
                    onMouseOut={hideTip}
                    key={index}
                  >
                    {isTextHighlight ? (
                      <Highlight
                        position={highlight.position}
                        comment={highlight.comment}
                        isScrolledTo={false}
                      />
                    ) : (
                      <AreaHighlight
                        highlight={highlight}
                        isScrolledTo={false}
                        onChange={() => {}}
                      />
                    )}
                  </Popup>
                );
              }}
            />
          );
        }}
      </PdfLoader>
    </div>
  );
};
```

#### 6.3 在 Chat 组件中使用
```typescript
// pages/Chat.tsx
import { useState } from 'react';
import { PDFViewer } from '@/components/PDFViewer';
import { useDocumentPreview, useChunkHighlights } from '@/hooks';

export const ChatPage = () => {
  const [selectedChunk, setSelectedChunk] = useState<any>(null);
  const [selectedDocId, setSelectedDocId] = useState<string>('');
  const [showPDF, setShowPDF] = useState(false);

  const { getDocumentUrl } = useDocumentPreview(selectedDocId);
  const { highlights, setWidthAndHeight } = useChunkHighlights(selectedChunk);

  const handleChunkClick = (chunk: any, docId: string) => {
    setSelectedChunk(chunk);
    setSelectedDocId(docId);
    setShowPDF(true);
  };

  return (
    <div style={{ display: 'flex', height: '100vh' }}>
      {/* Chat 区域 */}
      <div style={{ flex: 1 }}>
        <ChatMessages onChunkClick={handleChunkClick} />
      </div>

      {/* PDF 预览区域 */}
      {showPDF && selectedDocId && (
        <div style={{ flex: 1, borderLeft: '1px solid #ccc' }}>
          <button onClick={() => setShowPDF(false)}>关闭</button>
          <PDFViewer
            url={getDocumentUrl(selectedDocId)}
            highlights={highlights}
            onHeightChange={setWidthAndHeight}
          />
        </div>
      )}
    </div>
  );
};
```

### 后端部分

#### 6.4 Python/FastAPI 实现
```python
# api/document.py
from fastapi import APIRouter, Depends, HTTPException
from fastapi.responses import Response
from sqlalchemy.orm import Session
import os

router = APIRouter(prefix="/api/v1", tags=["documents"])

@router.get("/documents/{doc_id}/preview")
async def get_document_preview(
    doc_id: str,
    current_user: dict = Depends(get_current_user),  # 认证
    db: Session = Depends(get_db),
):
    """
    获取 PDF 二进制数据
    
    Args:
        doc_id: 文档 ID
    
    Returns:
        PDF 二进制数据
    """
    # 1️⃣ 查询数据库
    doc = db.query(Document).filter(
        Document.id == doc_id,
        Document.created_by == current_user.id,  # 权限检查
    ).first()
    
    if not doc:
        raise HTTPException(status_code=404, detail="Document not found")
    
    # 2️⃣ 从文件系统或云存储获取文件
    file_path = os.path.join(
        STORAGE_DIR,
        current_user.id,
        doc.file_path,
    )
    
    if not os.path.exists(file_path):
        raise HTTPException(status_code=404, detail="File not found")
    
    # 3️⃣ 读取二进制数据
    with open(file_path, 'rb') as f:
        file_data = f.read()
    
    # 4️⃣ 返回二进制响应
    return Response(
        content=file_data,
        media_type="application/pdf",  # Content-Type
        headers={
            "Content-Disposition": f'inline; filename="{doc.name}"',
            "Accept-Ranges": "bytes",  # 支持断点续传
        },
    )
```

## 第 7 步：关键知识点总结

### 7.1 为什么返回二进制而不是 Base64？

| 方案 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| **二进制** ✅ | 文件大小小 30%, 浏览器原生支持, 支持断点续传 | 需要正确设置 Content-Type | **PDF 渲染** ✅ |
| Base64 | 可以直接嵌入 HTML | 数据大 30%, 转码开销 | 小图片、数据 URI |
| URL 链接 | 前端简单 | 增加 HTTP 请求, 服务器负担 | 文件下载 |

### 7.2 HTTP Headers 的作用

```
Content-Type: application/pdf
  → 告诉浏览器这是 PDF 文件，使用 PDF 渲染器

Content-Disposition: inline; filename="document.pdf"
  → inline: 在浏览器中显示（不是下载）
  → filename: 如果用户保存，使用这个名称

Accept-Ranges: bytes
  → 支持 HTTP Range 请求（断点续传、快进）
  → 对大型 PDF 很重要
```

### 7.3 坐标系统

```
PDF 坐标系 (点数)          浏览器坐标系 (相对位置)
┌────────────────┐         ┌──────────────┐
│                │         │ (0, 0)       │
│  (100, 200)    │    →    │ (x: 0.12,    │
│  ↑ bbox        │         │  y: 0.17)    │
│  800×600 page  │         │              │
└────────────────┘         └──────────────┘

buildChunkHighlights 做的转换:
x_relative = bbox.x / page_width
y_relative = bbox.y / page_height
```

### 7.4 认证流程

```typescript
// 前端
const httpHeaders = {
  Authorization: `Bearer ${token}`,  // JWT token
};

fetch('/api/v1/documents/{id}/preview', {
  headers: httpHeaders,
})

// 后端
@app.get("/documents/{id}/preview")
@login_required  // 装饰器验证 token
async def preview(doc_id):
    # 此时 current_user 已经被验证
    pass
```

## 第 8 步：故障排查

### 问题 1: PDF 无法加载

**症状**:
```
PdfLoader 显示加载中，但一直不显示 PDF
```

**排查步骤**:
```typescript
// 1. 检查 URL
console.log('PDF URL:', url);
// 应该输出: /api/v1/documents/xxx/preview

// 2. 检查网络请求
// 打开浏览器 DevTools → Network 标签
// 看是否有 GET /api/v1/documents/xxx/preview 请求
// Status 应该是 200
// Content-Type 应该是 application/pdf

// 3. 检查认证
const httpHeaders = {
  Authorization: getAuthorization(),
};
console.log('Headers:', httpHeaders);
// 应该输出: { Authorization: 'Bearer eyJxxxxx' }

// 4. 检查后端日志
# 后端应该输出:
# GET /api/v1/documents/{doc_id}/preview
# 认证成功
# 文件找到
# 返回二进制数据
```

### 问题 2: 高亮位置不准确

**症状**:
```
PDF 显示了，但高亮的位置不对
```

**排查步骤**:
```typescript
// 1. 检查坐标数据
console.log('Chunk positions:', chunk.positions);
// 应该输出:
// [
//   {
//     page_number: 1,
//     bbox: { x: 100, y: 200, w: 150, h: 50 }
//   }
// ]

// 2. 检查 PDF 尺寸
const { width, height } = pdfDocument.getPage(1).getViewport({ scale: 1 });
console.log('PDF size:', width, height);
// 一般是: 612 × 792 (Letter size)

// 3. 检查转换后的高亮
console.log('Highlights:', highlights);
// 应该输出:
// [
//   {
//     position: {
//       pageNumber: 1,
//       x: 0.163,  // 100 / 612
//       y: 0.252,  // 200 / 792
//       width: 0.245,  // 150 / 612
//       height: 0.063   // 50 / 792
//     },
//     ...
//   }
// ]

// 4. 问题可能来自:
// - chunk.positions 坐标错误
// - PDF 有缩放
// - 页码不匹配
```

### 问题 3: 401 认证失败

**症状**:
```
Networking tab 显示 401 Unauthorized
```

**排查步骤**:
```typescript
// 1. 检查 token 是否存在
const token = localStorage.getItem('auth_token');
console.log('Token:', token);

// 2. 检查 token 是否过期
const decoded = JWT.decode(token);
console.log('Token expires at:', new Date(decoded.exp * 1000));

// 3. 重新获取 token
await refreshToken();

// 4. 后端检查
# 确保 login_required 装饰器正确
# 确保 Authorization header 被正确解析
print(request.headers.get('Authorization'))  # 应该输出: Bearer xxx
```

## 第 9 步：性能优化

### 9.1 懒加载 PDF.js Worker

```typescript
// 只在需要时加载 worker
const [workerReady, setWorkerReady] = useState(false);

useEffect(() => {
  // 动态加载 worker
  const script = document.createElement('script');
  script.src = '/pdfjs-dist/pdf.worker.min.js';
  script.onload = () => setWorkerReady(true);
  document.head.appendChild(script);
}, []);

if (!workerReady) return <div>Initializing...</div>;
```

### 9.2 缓存 PDF 数据

```typescript
const pdfCache = new Map<string, ArrayBuffer>();

const loadPDF = useCallback(async (docId: string) => {
  // 检查缓存
  if (pdfCache.has(docId)) {
    return pdfCache.get(docId);
  }
  
  // 获取数据
  const response = await fetch(`/api/v1/documents/${docId}/preview`);
  const buffer = await response.arrayBuffer();
  
  // 缓存
  pdfCache.set(docId, buffer);
  
  return buffer;
}, []);
```

### 9.3 虚拟化大 PDF

```typescript
// 只渲染当前可见的页面
<PdfHighlighter
  pdfDocument={pdfDocument}
  renderMode="canvas"  // 使用 canvas 而不是 SVG
  maxPages={10}        // 最多渲染 10 页
  highlights={highlights}
/>
```

## 第 10 步：完整测试

### 10.1 单元测试

```typescript
// __tests__/hooks/useDocumentPreview.test.ts
import { renderHook } from '@testing-library/react';
import { useDocumentPreview } from '@/hooks';

describe('useDocumentPreview', () => {
  it('应该生成正确的 URL', () => {
    const { result } = renderHook(() => useDocumentPreview('doc-123'));
    const url = result.current.getDocumentUrl('doc-123');
    
    expect(url).toBe('/api/v1/documents/doc-123/preview');
  });
});
```

### 10.2 集成测试

```typescript
// __tests__/components/PDFViewer.test.tsx
import { render, waitFor } from '@testing-library/react';
import { PDFViewer } from '@/components/PDFViewer';

describe('PDFViewer', () => {
  it('应该加载并渲染 PDF', async () => {
    const { container } = render(
      <PDFViewer
        url="/api/v1/documents/test-doc/preview"
        highlights={[
          {
            position: { pageNumber: 1, x: 0.1, y: 0.2, width: 0.3, height: 0.1 },
            comment: { emoji: '📌', text: 'Test highlight' },
          },
        ]}
      />
    );
    
    // 等待 PDF 加载
    await waitFor(
      () => {
        expect(container.querySelector('.pdfViewer')).toBeInTheDocument();
      },
      { timeout: 3000 }
    );
  });
});
```

### 10.3 E2E 测试

```typescript
// e2e/pdf-viewer.spec.ts
import { test, expect } from '@playwright/test';

test('应该显示 PDF 并高亮 chunks', async ({ page }) => {
  await page.goto('http://localhost:3000/chat');
  
  // 点击参考文献
  await page.click('[data-testid="reference-chunk-1"]');
  
  // 等待 PDF 加载
  await page.waitForSelector('.pdfViewer');
  
  // 验证高亮存在
  const highlights = await page.locator('.react-pdf-highlighter__highlight');
  expect(highlights).toHaveCount(1);
  
  // 验证 tooltip
  await highlights.first().hover();
  await page.waitForSelector('.react-pdf-highlighter__popup');
});
```

## 总结

### 核心要点

1. ✅ **Hook 层**: `useGetDocumentUrl` 生成 URL，`useGetChunkHighlights` 生成高亮数据
2. ✅ **组件层**: `PdfSheet` 容器组件，组织所有数据
3. ✅ **渲染层**: `PdfPreview` 使用 `react-pdf-highlighter` 渲染
4. ✅ **网络层**: 前端请求 `/api/v1/documents/{id}/preview`
5. ✅ **后端层**: 返回原始二进制 PDF 数据
6. ✅ **浏览器**: PDF.js 自动解析二进制数据并渲染

### 实现检查清单

- [ ] 后端 API 返回二进制数据（不是 Base64）
- [ ] 设置正确的 `Content-Type: application/pdf` header
- [ ] 设置 `Content-Disposition: inline` 允许在浏览器中显示
- [ ] 前端正确构建 URL
- [ ] 前端正确计算高亮坐标（坐标转换）
- [ ] 使用认证 token 进行请求
- [ ] 处理加载状态和错误
- [ ] 支持滚动到高亮位置

### 下一步

1. 复制代码到你的项目
2. 配置 PDF.js worker 路径
3. 测试 PDF 加载
4. 调整高亮样式
5. 添加更多交互功能（下载、打印等）
